In [ ]:
#Notebook description

#This notebook is being used to evaluate the techinical market conditions of a single asset and assess
#the appropriate strategy to take in order to maximize returns.

In [ ]:
#take all compuation functions and put them in a separate file

#simplify the date x axis on the percent drawdown chart

#default the zoom range to a comfortable range, and create a dropdown to select the time range for Volatility section


#Remove the VIX charting, its redudnant now that we have the volatility models

In [ ]:
# Block: load core libraries and instantiate helpers and plotters
# Load libraries
import logging
import warnings
import json
import os
import numpy as np
from scipy.stats import skew, kurtosis
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor
from plotly.subplots import make_subplots
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from Quantapp.visualization import Plotter, LineChartPlotter, CandleStickPlotter, BarChartPlotter, add_sigma_reference_lines, add_mean_reference_line, add_std_annotations, add_zone_annotation, add_horizontal_zone_trace, build_time_range_buttons, build_detail_visibility_mask, build_visibility_mask
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, Helper, Models, MomentumAnalytics, RiskDistributionAnalytics, RiskRelativeAnalytics, SeriesTransforms, calculate_zscore, calculate_max_drawdown, gini_coefficient, calculate_window_metrics
from Quantapp.data import MacroDataClient, normalize_benchmark_tickers, load_benchmark_data, align_series_to_common_index
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
logger.disabled = True
logger.propagate = False
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()


In [ ]:
#PARAMETERS (Adjust parmeters here)
# Define parameters for the analysis (Adjust these as needed)
interval = '1d'
period     = '20y'
risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days
#should be a string or None
ticker_str ='XLV'#ticker to analyze
benchmark_tickers = ['SPY']
length_of_plots = 20 #number of years of data to plot (after computing the period, this will be adjusted to the closest available data)
trading_strategy = 'position'  # Options: 'position', 'swing', or 'structural', # Determines the trading strategy to use for time frames


In [ ]:
# Block: organize structural, position, swing timeframe lists

# Define strategy timeframe profiles
TIMEFRAME_PROFILES = {
    'swing': {'short': 3, 'mid': 9, 'long': 21},
    'position': {'short': 21, 'mid': 50, 'long': 200},
    'structural': {'short': 200, 'mid': 500, 'long': 1000},
}

strategy = str(trading_strategy).strip().lower()
if strategy not in TIMEFRAME_PROFILES:
    raise ValueError(
        f"Invalid trading_strategy '{trading_strategy}'. "
        f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
    )

time_frame_map = TIMEFRAME_PROFILES[strategy]
time_frame_short = time_frame_map['short']
time_frame_mid = time_frame_map['mid']
time_frame_long = time_frame_map['long']

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block: display regime-change candlestick with Bollinger bands

#REGIME CHANGES: Candlestick with RSI and Bollinger Bands
candleStickPlotter.create_candlestick_fig(ticker, period='2y', bollinger_window=50, title="Candlestick with 50-Period Bollinger Bands")

In [ ]:
# Block: plot percentage drop from historical peaks

#REGIME CHANGES: Percentage Drop from Highest Peak
n = int(252 / 2)
qp.plot_percentage_drop(ticker, n=n, title=f'{ticker_str} Percentage Drop from Highest Peak')

In [ ]:
# Block: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))

momentum_diagnostics_context = momentum_analytics.build_momentum_window_diagnostics_context(
    close_series=ticker['Close'],
    window_sizes=window_sizes,
    highlight_windows=(7, 21, 50, 200),
    surface_years=10,
)

momentum_diagnostic_figs = lineChartPlotter.plot_momentum_window_diagnostics(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

momentum_diagnostic_figs['optimal_window'].show()
momentum_diagnostic_figs['optimal_window_histogram'].show()
momentum_diagnostic_figs['sharpe_mean_median'].show()
momentum_diagnostic_figs['volatility_mean_median'].show()
momentum_diagnostic_figs['sharpe_surface_3d'].show()


In [ ]:
# Block: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3]
)


In [ ]:
# Block: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
fig_ticker_Seasonality_Monthly.show()

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
fig_ticker_Seasonality_weekly.show()

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
fig_ticker_Seasonality_daily.show()

In [ ]:
# Block: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    ticker['Close'],
    window_pairs=window_pairs,
)

fig = lineChartPlotter.plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="50 vs 200",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.show()


In [ ]:
# Block: compute Sharpe/Sortino ratios and spreads

asset_close = ticker['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']


In [ ]:
# Block: plot Sharpe & Sortino efficiency with dropdown controls

long_window = time_frame_map.get('long')
default_term_label = f"{long_window}-day" if long_window is not None else max(term_config_map, key=lambda label: int(str(label).split('-')[0]))

fig = lineChartPlotter.plot_sharpe_sortino_comparison(
    term_config_map=term_config_map,
    ticker_label=ticker_str,
    default_label=default_term_label,
)
fig.show()


In [ ]:
# Block: combine risk-adjusted return and benchmark plots

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=time_frame_map,
    )

    default_term = 'long' if 'long' in time_frame_map else max(time_frame_map, key=time_frame_map.get)

    summary_fig = lineChartPlotter.plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_term=default_term,
    )
    summary_fig.show()

    detail_fig = lineChartPlotter.plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
        default_term=default_term,
    )
    detail_fig.show()
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block: analyze drawdown, skew, kurtosis, and Gini z-scores

dd_window_options = [21, 50, 200]

risk_distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=dd_window_options,
    default_window=200 if 200 in dd_window_options else max(dd_window_options),
)

fig = lineChartPlotter.plot_risk_distribution_zscores(
    metrics_by_window=risk_distribution_context['metrics_by_window'],
    window_options=risk_distribution_context['windows'],
    default_window=risk_distribution_context['default_window'],
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
#Idiosyncratic risk via Fama-French Factor Analysis
analysis_period = "max"
interval = "1d"
rolling_window = 252

ff_proxy = model.run_ff5_proxy_analysis(
    asset_ticker=ticker_str,
    period=analysis_period,
    interval=interval,
    window=rolling_window,
    auto_window=True,
    verbose=True,
)

prices_df = ff_proxy["proxy_prices"]
returns = ff_proxy["proxy_returns"]
factor_returns = ff_proxy["factor_returns_all"]
factor_returns_capm = ff_proxy["factor_returns_capm"]
factor_returns_ff3 = ff_proxy["factor_returns_ff3"]
factor_returns_ff5 = ff_proxy["factor_returns_ff5"]
factor_returns_ff5_custom = factor_returns_ff5.copy()
stock_returns = ff_proxy["stock_returns"]
rolling_results_ff5_custom = ff_proxy["rolling_results"]

#Plotting the Fama-French Factor Analysis Results
qp.plot_rolling_regression(rolling_results_ff5_custom, ticker_str, factor_returns_ff5_custom)
qp.plot_idiosyncratic_risk(rolling_results_ff5_custom, ticker_str)
